In [ ]:
# ==============================================================================
# SCRIPT FINAL: POTHOLE SEGMENTATION WINNING SOLUTION
# Architecture: U-Net + ResNet34 Backbone
# Features: 8-Way TTA, CoarseDropout, GridDistortion
# ==============================================================================

# 1. SETUP & LIBRARIES
# ------------------------------------------------------------------------------
try:
    import segmentation_models_pytorch as smp
except ImportError:
    !pip install segmentation_models_pytorch -q
    import segmentation_models_pytorch as smp

import os
import cv2
import numpy as np
import pandas as pd
import random
from glob import glob
from tqdm.notebook import tqdm # Pakai tqdm notebook biar bar loadingnya rapi
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [ ]:


# 2. KONFIGURASI (HYPERPARAMETERS)
# ------------------------------------------------------------------------------
class Config:
    SEED = 42
    IMG_SIZE = 256        # Ukuran input ke model
    BATCH_SIZE = 16       # Sesuaikan dengan VRAM (turunkan ke 8 jika OOM)
    EPOCHS = 30           # Tambah sedikit epoch karena ada CoarseDropout
    LR = 2e-4             # Learning Rate
    N_FOLDS = 5           # 5-Fold Cross Validation
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 2
    
    # GANTI PATH INI SESUAI LOKASI DATAMU
    TRAIN_IMG_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/train/images"
    TRAIN_MASK_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/train/mask"
    TEST_IMG_DIR = "/content/drive/MyDrive/ARA-ITS/dataset/test/images"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)

In [ ]:


# 3. UTILITIES (RLE & METRICS)
# ------------------------------------------------------------------------------
def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    # Fungsi RLE resmi dari panitia
    binary = (mask == pos_value).astype(np.uint8)
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

def dice_coef(y_pred, y_true, smooth=1e-7):
    y_pred = y_pred.sigmoid()
    y_pred = (y_pred > 0.5).float()
    intersection = (y_pred * y_true).sum()
    return (2. * intersection + smooth) / (y_pred.sum() + y_true.sum() + smooth)

In [ ]:
# 4. DATASET & ADVANCED AUGMENTATION
# ------------------------------------------------------------------------------
def get_transforms(data):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            
            # --- GEOMETRIC (Bentuk) ---
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=25, p=0.5),
            
            # --- DISTORTION (Tekstur Jalan) ---
            # Meniru jalan yang tidak rata/bergelombang
            A.OneOf([
                A.GridDistortion(num_steps=5, distort_limit=0.05, p=1.0),
                A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0)
            ], p=0.3),
            
            # --- COLOR (Pencahayaan) ---
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
                A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0), # Menonjolkan detail aspal
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
            ], p=0.5),
            
            # --- CUTOUT / DROPOUT (Ide Temanmu) ---
            # "Membagi" gambar dengan menghapus kotak-kotak kecil.
            # Memaksa model tidak manja; harus bisa menebak lubang meski sebagian tertutup.
            A.CoarseDropout(
                max_holes=8, max_height=32, max_width=32, 
                min_holes=4, min_height=8, min_width=8, 
                fill_value=0, mask_fill_value=0, p=0.5
            ),
            
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    elif data == 'valid':
        return A.Compose([
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

class PotholeDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.mask_paths:
            mask_path = self.mask_paths[idx]
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = np.where(mask > 127, 1.0, 0.0)
            
            if self.transform:
                augmented = self.transform(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']
            
            mask = mask.unsqueeze(0)
            return image, mask
        else:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, os.path.basename(img_path)

In [ ]:
# 5. LOSS FUNCTION & MODEL
# ------------------------------------------------------------------------------
class BCEDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(BCEDiceLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)
        inputs_sig = torch.sigmoid(inputs)
        inputs_flat = inputs_sig.view(-1)
        targets_flat = targets.view(-1)
        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs_flat.sum() + targets_flat.sum() + smooth)
        return 0.5 * bce_loss + 0.5 * dice_loss

In [ ]:
# 6. TRAINING LOOP (Jalankan ini untuk melatih model baru)
# ------------------------------------------------------------------------------
# Jika kamu sudah punya model tersimpan dan hanya ingin prediksi ulang,
# kamu bisa lewati blok ini dan langsung ke blok nomor 7.
# ------------------------------------------------------------------------------
img_paths = np.array(sorted(glob(f"{Config.TRAIN_IMG_DIR}/*")))
mask_paths = np.array(sorted(glob(f"{Config.TRAIN_MASK_DIR}/*")))

kf = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
best_models = []

print(f"Dataset Size: {len(img_paths)} images")
print(f"Mulai Training {Config.N_FOLDS}-Fold Cross Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(img_paths)):
    print(f"\n{'='*15} Fold {fold+1}/{Config.N_FOLDS} {'='*15}")
    
    train_ds = PotholeDataset(img_paths[train_idx], mask_paths[train_idx], transform=get_transforms('train'))
    val_ds = PotholeDataset(img_paths[val_idx], mask_paths[val_idx], transform=get_transforms('valid'))
    
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    # MODEL: U-Net with ResNet34 Backbone
    model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = BCEDiceLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS, eta_min=1e-6)
    
    best_dice = 0.0
    best_model_path = f"unet_resnet34_fold{fold}.pth"
    
    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        for images, masks in tqdm(train_loader, desc=f"Ep {epoch+1} Train", leave=False):
            images, masks = images.to(Config.DEVICE), masks.to(Config.DEVICE).float()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        model.eval()
        val_dice_score = 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Ep {epoch+1} Val", leave=False):
                images, masks = images.to(Config.DEVICE), masks.to(Config.DEVICE).float()
                outputs = model(images)
                val_dice_score += dice_coef(outputs, masks).item()
        
        scheduler.step()
        train_loss /= len(train_loader)
        val_dice_score /= len(val_loader)
        
        if val_dice_score > best_dice:
            best_dice = val_dice_score
            torch.save(model.state_dict(), best_model_path)
            print(f"Epoch {epoch+1}: Val Dice IMPROVED to {best_dice:.4f} (Loss: {train_loss:.4f})")
        else:
            print(f"Epoch {epoch+1}: Val Dice {val_dice_score:.4f} (Loss: {train_loss:.4f})")
            
    best_models.append(best_model_path)
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()

In [ ]:
# 7. INFERENCE with CUSTOM 8-WAY TTA (Strategy Discussion)
# ------------------------------------------------------------------------------
print(f"\nMemulai Prediksi dengan 8-Way TTA (Grandmaster Strategy)...")

test_paths = sorted(glob(f"{Config.TEST_IMG_DIR}/*"))
test_ds = PotholeDataset(test_paths, transform=get_transforms('valid'))
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

# Load Trained Models
models = []
# Pastikan best_models terisi. Jika runtime restart, isi manual:
# best_models = ['unet_resnet34_fold0.pth', 'unet_resnet34_fold1.pth', 'unet_resnet34_fold2.pth', 'unet_resnet34_fold3.pth', 'unet_resnet34_fold4.pth']
for path in best_models:
    m = smp.Unet(encoder_name="resnet34", in_channels=3, classes=1)
    m.load_state_dict(torch.load(path))
    m.to(Config.DEVICE)
    m.eval()
    models.append(m)

# FUNGSI TTA SESUAI REQUEST (1-8 URUTAN)
def tta_inference_custom(model, images):
    batch_size, c, h, w = images.shape
    final_pred = torch.zeros((batch_size, 1, h, w)).to(Config.DEVICE)
    
    # FORMAT: (Nama Langkah, Flip Horizontal?, Rotasi k-kali)
    # k=0(0deg), k=1(90deg), k=2(180deg), k=3(270deg)
    
    steps = [
        # 1. Original
        ("Original", False, 0),
        # 2-4. Rotasi Saja
        ("Rot90",    False, 1),
        ("Rot180",   False, 2),
        ("Rot270",   False, 3),
        # 5. Flip Horizontal
        ("Flip",     True, 0),
        # 6-8. Flip Horizontal + Rotasi
        ("Flip+Rot90",  True, 1),
        ("Flip+Rot180", True, 2),
        ("Flip+Rot270", True, 3),
    ]
    
    # Kita iterasi sesuai urutan 1 sampai 8
    for name, do_flip, k_rot in steps:
        # A. AUGMENTASI (Siapkan Soal Ujian)
        aug_img = images.clone()
        
        # Urutan: Flip dulu, baru Rotasi (Konsisten)
        if do_flip:
            aug_img = torch.flip(aug_img, [3]) # Dim 3 = Width
        if k_rot > 0:
            aug_img = torch.rot90(aug_img, k_rot, [2, 3]) # Rotasi plane H,W
            
        # B. PREDIKSI (Jawab Soal)
        with torch.no_grad():
            pred = model(aug_img)
            pred = torch.sigmoid(pred)
        
        # C. INVERSE / KEMBALIKAN (Cocokkan Jawaban ke Posisi Asli)
        # Urutan Inverse: Un-Rotate dulu, baru Un-Flip
        if k_rot > 0:
            # Jika diputar 1x (90), balikkan dengan putar 3x (270) atau -1
            pred = torch.rot90(pred, 4 - k_rot, [2, 3])
            
        if do_flip:
            pred = torch.flip(pred, [3])
            
        final_pred += pred
        
    # Rata-rata dari 8 jawaban
    return final_pred / len(steps)

def remove_small_blobs(mask, min_size=30):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    new_mask = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            new_mask[labels == i] = 255
    return new_mask

# MAIN PREDICTION LOOP
submission_data = []

for images, ids in tqdm(test_loader, desc="Predicting"):
    images = images.to(Config.DEVICE)
    
    # Ensemble Logic:
    # 5 Model x 8 TTA Views = 40 Prediksi per gambar
    ensemble_pred = torch.zeros((images.size(0), 1, Config.IMG_SIZE, Config.IMG_SIZE)).to(Config.DEVICE)
    
    for model in models:
        ensemble_pred += tta_inference_custom(model, images)
        
    ensemble_pred /= len(models)
    ensemble_pred = ensemble_pred.cpu().numpy()
    
    for i in range(len(ids)):
        image_id = ids[i]
        pred_mask = ensemble_pred[i, 0]
        
        # 1. Resize ke Ukuran Asli Gambar (WAJIB)
        original_img = cv2.imread(os.path.join(Config.TEST_IMG_DIR, image_id))
        h_orig, w_orig = original_img.shape[:2]
        pred_mask = cv2.resize(pred_mask, (w_orig, h_orig))
        
        # 2. Thresholding
        # Dengan 40 votes, threshold 0.5 sangat solid
        binary_mask = (pred_mask > 0.5).astype(np.uint8) * 255
        
        # 3. Cleanup Noise
        clean_mask = remove_small_blobs(binary_mask, min_size=30)
        
        # 4. RLE Encoding
        if clean_mask.sum() == 0:
            rle_str = ""
        else:
            rle_str = encode_rle(clean_mask)
        
        submission_data.append([image_id, rle_str])



In [ ]:
# 8. SAVE SUBMISSION
df_sub = pd.DataFrame(submission_data, columns=['ImageId', 'rle'])
df_sub.to_csv('submission_winning_tta_final.csv', index=False)
print("Selesai! Submission tersimpan: submission_winning_tta_final.csv")
print(df_sub.head())